# Thesis Note — Milestone 8.1: Natural Missingness Audit

**Project:** Self-Supervised Multimodal Representation Learning for Robust Dental Diagnosis with Naturally Missing Radiographs: A Study on the COde Dataset
**Milestone:** 8.1 — Natural Missingness Audit
**Status:** Completed
**Date:** 2026-08-31

---

## 1. Purpose

The purpose of this sub-milestone was to characterize the naturally occurring missing-modality patterns in the authoritative six-label patient-level benchmark dataset before designing or evaluating any missing-modality method.

This audit was performed to establish:

* how frequently each modality is missing,
* which combinations of available modalities occur naturally,
* how these patterns are distributed across train, validation, and test splits,
* how many labeled visits are available within each pattern,
* and whether the existing patient-level split remains valid.

No samples were removed, no modality was imputed, and the authoritative dataset was not modified.

---

## 2. Authoritative Dataset

The analysis used the finalized six-label patient-level dataset:

```text
results/six_label_patient_level_dataset/labeled_dataset.csv
```

This dataset is derived from the finalized patient-level dataset and preserves the authoritative patient-level split.

### Dataset statistics

| Property            | Value |
| ------------------- | ----: |
| Total visits        | 8,775 |
| Total patients      | 4,800 |
| Labels              |     6 |
| Patient-level split |   Yes |
| Dataset modified    |    No |

The six target labels are:

1. Caries
2. Gingivitis
3. Malocclusion
4. Pulpitis
5. Tooth Loss
6. Tooth Structure Loss

---

## 3. Modality Definition

Three modalities were considered:

* **Image:** intraoral photographs
* **X-ray:** dental radiographs
* **Text:** clinical examination text

A modality was considered missing when no valid representation of that modality was available for the corresponding visit.

For clinical text, empty values such as empty strings, `NaN`, `None`, `null`, `NA`, and `N/A` were treated as missing.

The audit preserved missingness as an observed property of the original data. No imputation was performed.

---

## 4. Natural Modality Availability

The dataset contains the following modality availability:

| Modality | Present | Missing | Missing Rate |
| -------- | ------: | ------: | -----------: |
| Image    |   8,772 |       3 |        0.03% |
| X-ray    |   4,256 |   4,519 |       51.50% |
| Text     |   8,568 |     207 |        2.36% |

The dominant source of missingness is therefore the X-ray modality.

Importantly, X-ray missingness is not a rare edge case: more than half of all visits do not contain a radiograph.

---

## 5. Natural Missing-Modality Patterns

Each visit was assigned a modality-presence pattern using:

```text
111 = Image + X-ray + Text
110 = Image + X-ray
101 = Image + Text
100 = Image only
011 = X-ray + Text
001 = Text only
```

The observed patterns were:

| Code | Modality Pattern     | Visits | Patients | Percentage |
| ---- | -------------------- | -----: | -------: | ---------: |
| 111  | Image + X-ray + Text |  4,137 |    2,803 |     47.15% |
| 110  | Image + X-ray        |    118 |      117 |      1.34% |
| 101  | Image + Text         |  4,428 |    2,817 |     50.46% |
| 100  | Image only           |     89 |       85 |      1.01% |
| 011  | X-ray + Text         |      1 |        1 |      0.01% |
| 001  | Text only            |      2 |        1 |      0.02% |

There were therefore six naturally occurring modality patterns.

---

## 6. Mapping to Missing-Modality Scenarios

The four predefined experimental scenarios can be mapped to the naturally occurring patterns as follows:

| Scenario          | Available Modalities | Natural Pattern |
| ----------------- | -------------------- | --------------- |
| A — Complete      | Image + X-ray + Text | 111             |
| B — X-ray Missing | Image + Text         | 101             |
| C — Image Only    | Image                | 100             |
| D — Text Missing  | Image + X-ray        | 110             |

The remaining patterns (`011` and `001`) occur extremely rarely and are therefore treated as **other natural patterns**, rather than primary experimental scenarios.

This distinction prevents the main experiments from being dominated by extremely small populations.

---

## 7. Label Coverage Within Missingness Patterns

The number of visits containing at least one of the six target labels was also examined.

| Pattern              | Visits | Labeled Visits | Label Coverage |
| -------------------- | -----: | -------------: | -------------: |
| Image + X-ray + Text |  4,137 |          3,227 |         78.00% |
| Image + X-ray        |    118 |              1 |          0.85% |
| Image + Text         |  4,428 |          3,407 |         76.94% |
| Image only           |     89 |              3 |          3.37% |
| X-ray + Text         |      1 |              1 |        100.00% |
| Text only            |      2 |              2 |        100.00% |

The primary complete and X-ray-missing patterns contain substantial numbers of labeled visits.

However, the Image + X-ray and Image-only patterns have extremely low six-label coverage. Therefore, these patterns should be interpreted carefully in downstream performance evaluation.

In particular, their small labeled populations make them unsuitable for strong standalone conclusions without additional controlled experiments.

---

## 8. Distribution Across Patient-Level Splits

The natural missingness patterns were also inspected separately within the authoritative train, validation, and test splits.

### Train

| Pattern       | Visits |
| ------------- | -----: |
| Complete      |  2,896 |
| X-ray Missing |  3,097 |
| Text Missing  |     75 |
| Image Only    |     58 |
| X-ray + Text  |      1 |
| Text Only     |      2 |

Total train visits: **6,129**

### Validation

| Pattern       | Visits |
| ------------- | -----: |
| Complete      |    616 |
| X-ray Missing |    672 |
| Text Missing  |     26 |
| Image Only    |     16 |

Total validation visits: **1,330**

### Test

| Pattern       | Visits |
| ------------- | -----: |
| Complete      |    625 |
| X-ray Missing |    659 |
| Text Missing  |     17 |
| Image Only    |     15 |

Total test visits: **1,316**

The dominant two patterns remain consistent across all three splits:

* Complete multimodal: approximately 46–47%
* Image + Text with X-ray missing: approximately 50%

This is important because it shows that X-ray missingness is not concentrated exclusively in one split.

---

## 9. Patient-Level Split Validation

The audit verified that the existing patient-level split remains valid.

Validation results:

```text
patient_level_split: PASS
visit_uniqueness: PASS
authoritative_six_label_dataset: PASS
```

The audit did not regenerate or modify the train/validation/test assignment.

Therefore, all subsequent Milestone 8 experiments must continue to use the existing patient-level split.

---

## 10. Important Findings

The audit establishes several important facts for the next experiments.

### 10.1 X-ray missingness is the dominant missing-modality problem

There are:

```text
4,519 visits with missing X-ray
```

representing:

```text
51.50% of all visits
```

Therefore, missing X-ray is the most important naturally occurring missing-modality condition in the COde dataset.

### 10.2 The X-ray-missing scenario has a large evaluation population

The `Image + Text` pattern contains:

```text
4,428 visits
```

and is therefore large enough to support meaningful evaluation.

This makes Scenario B — X-ray Missing — the primary natural missing-modality scenario.

### 10.3 Complete cases are only a subset of the dataset

Only:

```text
4,137 / 8,775 = 47.15%
```

of visits contain all three modalities.

This provides an important motivation for investigating missing-modality robustness: a complete-case multimodal approach cannot directly use more than half of the visits.

### 10.4 Rare missingness patterns require caution

Missing text and missing image cases exist naturally, but their labeled populations are much smaller.

Therefore, performance on these cases should not be used alone to make strong conclusions.

For these scenarios, controlled missingness experiments will be important to obtain sufficiently large and balanced evaluation populations.

---

## 11. Methodological Decision for Milestone 8

The results support a two-part evaluation strategy for missing-modality robustness.

### Part 1 — Natural Missingness

Use the naturally occurring missingness in the COde dataset, particularly:

```text
Complete:
Image + X-ray + Text

X-ray Missing:
Image + Text
```

This measures robustness under the real missingness distribution of the dataset.

### Part 2 — Controlled Missingness

Artificially mask modalities in a controlled evaluation setting while preserving the same underlying samples.

This will allow fair comparison between modality conditions when natural missingness produces very different sample sizes.

Controlled missingness must not alter the authoritative dataset or patient-level split.

---

## 12. What This Sub-Milestone Does Not Do

Milestone 8.1 is an **audit only**.

It does not:

* train a new model,
* modify the SSL encoders,
* modify the Fusion architecture,
* impute missing modalities,
* remove incomplete samples,
* regenerate the patient-level split,
* compare model performance,
* or claim robustness.

The purpose is to establish the experimental population and missingness distribution before model evaluation.

---

## 13. Outputs

The following artifacts were generated:

```text
results/milestone8_missing_modality/
└── 01_natural_missingness/
    ├── natural_missingness_summary.json
    └── modality_patterns_by_split.csv
```

The implementation is located under:

```text
src/missing_modality/
```

The authoritative six-label dataset remains:

```text
results/six_label_patient_level_dataset/labeled_dataset.csv
```

---

## 14. Conclusion

Milestone 8.1 confirms that modality missingness is a substantial property of the COde dataset rather than an artificially constructed problem.

The most important observation is that **51.50% of visits are naturally missing X-ray data**, while the complete multimodal population represents only **47.15%** of visits.

Consequently, the next stage of Milestone 8 will evaluate whether the existing SSL-based multimodal fusion model can maintain diagnostic performance when one or more modalities are unavailable.

The central question for the next experiment is therefore not whether missingness exists, but:

> **How much performance does the existing multimodal model lose when a modality is unavailable, and can a missing-modality-aware fusion strategy reduce this degradation?**

---

**Milestone 8.1 Status: COMPLETED**

**Next:** Milestone 8.2 — Missing-Modality Baseline Evaluation


# Milestone 8.2 — Controlled Missing-Modality Robustness Evaluation

## 3. Model Architecture

The existing **SSL Fusion — Main** model uses the following architecture:

```text
Photograph representation: 2048 → 512
Radiograph representation: 2048 → 512
Text representation:       768  → 512

                    ↓

              Concatenation

                    ↓

                  1536

                    ↓

              Fusion MLP

                    ↓

                   512

                    ↓

             6-label output
```

### Checkpoint

```text
results/fusion/main/best_model.pt
```

The model was trained previously during **Milestone 7.3**.

### Important Protocol Constraint

No training or fine-tuning was performed during this experiment.

Therefore:

* The model architecture was unchanged.
* The model parameters were unchanged.
* The existing checkpoint was used directly.
* The test set was not used for model selection.

This ensures that the experiment measures the **robustness of the existing fusion model** rather than the effect of additional training.

---

## 4. Controlled Test Population

The evaluation uses the **same complete-case test population** used by the original Milestone 7.3 Fusion experiment.

**Number of test samples:**

```text
633
```

Every sample in this population originally contains:

```text
Photograph + Radiograph + Clinical Text
```

The same 633 samples were evaluated under every scenario.

This is important because it ensures that differences between scenarios are caused by the **missing-modality condition** rather than by changes in the evaluated population.

---

## 5. Controlled Missing-Modality Protocol

Missing modalities were simulated at the **representation level**.

For a missing modality, its SSL representation was replaced by a **zero vector with the same dimensionality** as the original representation.

No data imputation was performed.

### Example

**Complete:**

```text
Image representation
        +
X-ray representation
        +
Text representation
```

**Missing X-ray:**

```text
Image representation
        +
Zero X-ray representation
        +
Text representation
```

This allows the existing fusion model to be evaluated under identical samples while systematically removing modality information.

---

## 6. Evaluation Scenarios

Four controlled scenarios were evaluated.

### Scenario A — Complete

**Input:**

```text
Image + X-ray + Text
```

**Missing modalities:**

```text
None
```

This is the reference condition.

---

### Scenario B — Missing X-ray

**Input:**

```text
Image + Text
```

**Missing modality:**

```text
X-ray
```

The radiograph representation was replaced with a zero vector.

---

### Scenario C — Missing Text

**Input:**

```text
Image + X-ray
```

**Missing modality:**

```text
Clinical Text
```

The text representation was replaced with a zero vector.

---

### Scenario D — Missing Multiple Modalities

**Input:**

```text
Image Only
```

**Missing modalities:**

```text
X-ray + Text
```

Both corresponding representations were replaced with zero vectors.

---

## 7. Results

The results were:

| Scenario                 | Input Modalities     | Samples | Macro F1 | Micro F1 |  AUROC | Accuracy |
| ------------------------ | -------------------- | ------: | -------: | -------: | -----: | -------: |
| A — Complete             | Image + X-ray + Text |     633 |   0.7676 |   0.8760 | 0.9646 |   0.8120 |
| B — Missing X-ray        | Image + Text         |     633 |   0.6751 |   0.8260 | 0.9609 |   0.7378 |
| C — Missing Text         | Image + X-ray        |     633 |   0.3559 |   0.5366 | 0.7508 |   0.4850 |
| D — Missing X-ray + Text | Image Only           |     633 |   0.2441 |   0.2606 | 0.7200 |   0.2006 |

---

## 8. Performance Degradation

Performance degradation was calculated relative to the complete-case condition.

### Macro F1

| Scenario             | Macro F1 |   Drop |
| -------------------- | -------: | -----: |
| Complete             |   0.7676 | 0.0000 |
| Missing X-ray        |   0.6751 | 0.0925 |
| Missing Text         |   0.3559 | 0.4118 |
| Missing X-ray + Text |   0.2441 | 0.5236 |

The largest degradation occurs when **Text is unavailable**.

Removing X-ray produces a substantially smaller degradation.

Removing both X-ray and Text produces the largest overall performance loss.

---

### Micro F1

| Scenario             | Micro F1 |   Drop |
| -------------------- | -------: | -----: |
| Complete             |   0.8760 | 0.0000 |
| Missing X-ray        |   0.8260 | 0.0501 |
| Missing Text         |   0.5366 | 0.3395 |
| Missing X-ray + Text |   0.2606 | 0.6155 |

The same pattern is observed for Micro F1.

---

### AUROC

| Scenario             |  AUROC |   Drop |
| -------------------- | -----: | -----: |
| Complete             | 0.9646 | 0.0000 |
| Missing X-ray        | 0.9609 | 0.0037 |
| Missing Text         | 0.7508 | 0.2138 |
| Missing X-ray + Text | 0.7200 | 0.2447 |

Interestingly, missing X-ray causes only a **very small AUROC reduction**, despite the decrease in F1.

---

## 9. Interpretation

The controlled experiment demonstrates that the existing **SSL Fusion — Main** model is **not robust to missing modalities**.

The degradation is highly modality-dependent.

### Missing X-ray

When the radiograph representation is removed:

```text
Macro F1:
0.7676 → 0.6751

Drop:
0.0925
```

The model retains relatively strong discriminative ability.

This is consistent with the earlier dataset audit showing that radiographs are the naturally scarce modality in COde.

---

### Missing Text

When clinical text is removed:

```text
Macro F1:
0.7676 → 0.3559

Drop:
0.4118
```

This represents a substantial degradation.

This finding is also consistent with the previous SSL downstream experiments, where the text modality produced the strongest single-modality performance.

Therefore, the current fusion model appears to depend strongly on the information contained in the **clinical text representation**.

---

### Multiple Missing Modalities

When both X-ray and Text are removed:

```text
Macro F1:
0.7676 → 0.2441

Drop:
0.5236
```

The model performs poorly when only the image representation remains.

This demonstrates that the current fusion architecture cannot simply be expected to remain robust when its inputs become incomplete.

---

## 10. Main Finding

The most important finding of **Milestone 8.2** is:

> **The existing SSL Fusion — Main model experiences substantial performance degradation under missing-modality conditions, particularly when clinical text is unavailable.**

Therefore, simply applying the existing complete-case fusion model to incomplete multimodal inputs is insufficient.

This provides direct experimental motivation for the next stage of the research.

---

## 11. Relationship to Natural Missingness

The previous **Milestone 8.1** audit showed that natural missingness is substantial in the COde dataset.

The most important naturally occurring patterns were:

| Modality Pattern     | Visits | Percentage |
| -------------------- | -----: | ---------: |
| Image + X-ray + Text |  4,137 |     47.15% |
| Image + Text         |  4,428 |     50.46% |
| Image + X-ray        |    118 |      1.34% |
| Image Only           |     89 |      1.01% |

Therefore, the controlled experiment is particularly relevant to the real dataset.

The most common natural pattern is:

```text
Image + Text
X-ray Missing
```

which corresponds directly to **Scenario B**.

However, the controlled experiment is intentionally performed on the **fixed complete-case test population** so that the effect of missingness can be isolated without confounding the comparison by changing the evaluated samples.

Natural missingness will be evaluated separately in a later sub-milestone.

---

## 12. Research Implication

Milestone 8.2 establishes the **baseline failure mode** that the proposed missing-modality method must address.

The goal of the next stage is not necessarily to obtain a higher complete-case Macro F1 than the existing Main Fusion model.

Instead, the primary objective is:

> **Reduce performance degradation when one or more modalities are missing.**

A successful robust model should therefore preserve substantially more of its performance across:

```text
Complete
      ↓
Missing X-ray
      ↓
Missing Text
      ↓
Missing Multiple Modalities
```

while maintaining a comparable complete-case performance.

---

## 13. Next Step — Milestone 8.3

The next stage will design and train a **Missing-Modality Robust Fusion** model.

Unlike Milestone 8.2, this stage will involve model training.

The training protocol will use the **authoritative six-label patient-level dataset** and its existing patient-level split.

The training process will expose the model to different modality-availability patterns so that it learns to operate when one or more modalities are unavailable.

The resulting model will then be evaluated under the same controlled scenarios used here.

The primary comparison will therefore be:

```text
Existing Main Fusion
        vs
Missing-Modality Robust Fusion
```

with particular attention to:

```text
Performance under missing modalities
        and
Performance degradation from Complete
```

---

## 14. Validation and Reproducibility

The following protocol checks passed:

| Check                                     | Status                     |
| ----------------------------------------- | -------------------------- |
| Complete-case test population             | PASS                       |
| Same population across all scenarios      | PASS                       |
| Patient-level split                       | Inherited from Milestone 7 |
| Test set used for model selection         | NO                         |
| Model modified                            | NO                         |
| Model retrained                           | NO                         |
| Fine-tuning performed                     | NO                         |
| Imputation performed                      | NO                         |
| Zero-vector replacement used consistently | YES                        |

---

## 15. Conclusion

Milestone 8.2 successfully established a **controlled robustness baseline** for the existing SSL Fusion — Main model.

The complete-case performance was:

```text
Macro F1 = 0.7676
Micro F1 = 0.8760
AUROC    = 0.9646
Accuracy = 0.8120
```

Under missing X-ray conditions, Macro F1 decreased by only:

```text
0.0925
```

whereas missing Text caused a much larger decrease of:

```text
0.4118
```

and missing both X-ray and Text caused a decrease of:

```text
0.5236
```

These results demonstrate a clear robustness limitation in the existing fusion approach.

Consequently, the next stage will focus on designing a fusion model that **explicitly learns to handle missing modalities** rather than assuming that all modalities are always available.


# Milestone 8.3 — Robust Multimodal Fusion under Missing Modalities

## 8.3.1 Objective

The objective of Milestone 8.3 was to evaluate whether the frozen SSL representations can support a multimodal fusion classifier that remains robust when one or more modalities are unavailable at inference time.

This experiment directly targets the missing-modality problem identified in the COde dataset, where radiographs are substantially less available than photographs and clinical text.

The experiment extends the frozen-representation fusion framework developed in Milestone 7 by introducing **modality dropout during fusion training**.

The SSL encoders remain frozen throughout this experiment. Therefore, any robustness observed in this milestone is attributable to the fusion/classification stage rather than additional adaptation of the underlying SSL encoders.

---

## 8.3.2 Experimental Protocol

### Input representations

The experiment uses the SSL representations extracted during Milestone 7.

| Modality      | Representation dimension |
| ------------- | -----------------------: |
| Photograph    |                     2048 |
| Radiograph    |                     2048 |
| Clinical Text |                      768 |

Only the complete-case multimodal population used for the Milestone 7 fusion experiments was used as the source population.

The representation root was:

```text
results/fusion/ssl_representations
```

The population sizes were:

| Split      | Samples |
| ---------- | ------: |
| Train      |    2935 |
| Validation |     627 |
| Test       |     633 |

The test population was therefore kept identical to the previous frozen-SSL fusion experiment, allowing direct comparison across fusion approaches.

---

## 8.3.3 Robust Fusion Architecture

A dedicated `RobustFusion` classifier was trained on the frozen SSL representations.

Each modality representation is projected into a common 512-dimensional space:

```text
Photograph       2048 → 512
Radiograph       2048 → 512
Clinical Text     768 → 512
```

The projected representations are combined together with an explicit modality-presence mask.

The mask contains three binary values in the following order:

```text
[image, radiograph, text]
```

For example:

```text
[1, 1, 1]  → all modalities available
[0, 1, 1]  → image missing
[1, 0, 1]  → radiograph missing
[1, 1, 0]  → text missing
```

The explicit mask allows the classifier to distinguish between a genuine representation and a representation that has been intentionally removed because the corresponding modality is unavailable.

---

## 8.3.4 Modality Dropout during Training

During each training batch, modality availability was sampled dynamically.

Each modality was independently dropped with probability:

```text
p = 0.30
```

Thus, the model was exposed during training to multiple modality-availability configurations rather than only complete multimodal input.

Possible states include:

```text
Image + Radiograph + Text
Image + Text
Image + Radiograph
Image only
Radiograph + Text
Radiograph only
Text only
```

The all-missing state was explicitly prevented.

This training strategy was designed to encourage the fusion classifier to remain functional when one or more modalities are unavailable.

---

## 8.3.5 Training Configuration

| Parameter                         | Value               |
| --------------------------------- | ------------------- |
| Seed                              | 42                  |
| Device                            | CUDA                |
| Batch size                        | 64                  |
| Epochs                            | 50                  |
| Learning rate                     | 1e-4                |
| Weight decay                      | 1e-4                |
| Classifier dropout                | 0.30                |
| Modality dropout probability      | 0.30                |
| Optimizer                         | AdamW               |
| Loss                              | BCEWithLogitsLoss   |
| Model selection metric            | Validation Macro F1 |
| SSL encoders                      | Frozen              |
| Test set used for model selection | No                  |

The best checkpoint was selected exclusively according to validation Macro F1.

The resulting best checkpoint was obtained at:

```text
Epoch: 42
Validation Macro F1: 0.7347
```

Training artifacts were stored under:

```text
results/milestone8_missing_modality/04_robust_fusion_training/
```

including:

```text
best_model.pt
config.json
history.json
```

---

# 8.3.6 Missing-Modality Test Protocol

After training, the best validation checkpoint was evaluated on the fixed test population of 633 complete-case samples.

Unlike training, modality availability was deterministic during testing.

Seven non-empty modality configurations were evaluated:

1. Complete multimodal input
2. Image missing
3. Radiograph missing
4. Text missing
5. Image + radiograph missing
6. Image + text missing
7. Radiograph + text missing

The test set was not used during training or checkpoint selection.

The evaluation metrics were:

* Macro F1
* Micro F1
* AUROC
* Accuracy at threshold 0.5

---

# 8.3.7 Test Results

| Scenario                   | Mask      |   Macro F1 | Micro F1 |      AUROC | Accuracy |
| -------------------------- | --------- | ---------: | -------: | ---------: | -------: |
| Complete                   | `[1,1,1]` | **0.7756** |   0.8685 |     0.9658 |   0.8025 |
| Image missing              | `[0,1,1]` | **0.7853** |   0.8652 |     0.9657 |   0.7978 |
| Radiograph missing         | `[1,0,1]` |     0.6890 |   0.8470 | **0.9670** |   0.7741 |
| Text missing               | `[1,1,0]` |     0.5251 |   0.7261 |     0.8695 |   0.6477 |
| Image + Radiograph missing | `[0,0,1]` |     0.7110 |   0.8458 |     0.9622 |   0.7678 |
| Image + Text missing       | `[0,1,0]` |     0.4066 |   0.6326 |     0.8197 |   0.5671 |
| Radiograph + Text missing  | `[1,0,0]` |     0.4039 |   0.6918 |     0.8342 |   0.5972 |

---

# 8.3.8 Observations

### 1. Image removal caused minimal degradation

The complete-case Macro F1 was:

```text
0.7756
```

while removing the photograph modality resulted in:

```text
0.7853
```

The difference is approximately:

```text
+0.0096 Macro F1
```

Therefore, the model did not exhibit a meaningful performance degradation when the photograph modality was unavailable.

This suggests that, within this experimental setting, the information provided by the remaining radiograph and clinical-text representations was sufficient to maintain performance.

The slightly higher score under image removal should not be interpreted as evidence that missing images are beneficial. It is more appropriately interpreted as evidence that the robust model can tolerate the absence of the image modality.

---

### 2. Radiograph removal produced a moderate performance reduction

Removing the radiograph modality reduced Macro F1 from:

```text
0.7756 → 0.6890
```

corresponding to an approximate decrease of:

```text
8.66 percentage points
```

Micro F1 also decreased from:

```text
0.8685 → 0.8470
```

However, AUROC remained essentially unchanged:

```text
0.9658 → 0.9670
```

This indicates that although threshold-dependent classification performance was reduced, the model retained strong ranking ability under the radiograph-missing condition.

---

### 3. Clinical text was the most influential modality

Removing clinical text resulted in a substantially larger degradation:

```text
Macro F1:
0.7756 → 0.5251
```

which corresponds to a decrease of approximately:

```text
25.05 percentage points
```

AUROC also decreased substantially:

```text
0.9658 → 0.8695
```

Compared with image and radiograph removal, the absence of clinical text therefore had the largest impact on predictive performance.

This observation is consistent with the strong performance previously observed for the text modality in downstream evaluation.

---

### 4. Text-only inference remained comparatively strong

When both image and radiograph were removed, leaving only clinical text:

```text
Macro F1 = 0.7110
Micro F1 = 0.8458
AUROC = 0.9622
```

Despite using only one modality, the model retained relatively strong performance.

This provides further evidence that clinical text contains substantial predictive information for the selected diagnostic labels.

---

### 5. Image-only and radiograph-only configurations were substantially weaker

When clinical text was unavailable, performance decreased considerably.

Image-only:

```text
Macro F1 = 0.4039
AUROC = 0.8342
```

Radiograph-only:

```text
Macro F1 = 0.4066
AUROC = 0.8197
```

Thus, the model was substantially more dependent on clinical text than on either visual modality.

---

# 8.3.9 Interpretation

The results demonstrate that modality dropout training produces a fusion classifier capable of operating under multiple modality-availability conditions without retraining separate models for each scenario.

The most important observation is the asymmetric effect of modality removal.

The approximate Macro F1 degradation relative to the complete-case condition was:

| Missing condition          | Macro F1 change |
| -------------------------- | --------------: |
| Image missing              |    **+0.97 pp** |
| Radiograph missing         |    **−8.66 pp** |
| Text missing               |   **−25.05 pp** |
| Image + Radiograph missing |    **−6.47 pp** |
| Image + Text missing       |   **−36.90 pp** |
| Radiograph + Text missing  |   **−37.18 pp** |

The results therefore suggest that the three modalities do not contribute equally to the final diagnostic prediction.

In particular:

```text
Clinical Text > Radiograph > Photograph
```

appears to be the approximate ordering of their importance in this experimental configuration.

However, this ordering should be interpreted as an empirical observation from the current dataset and model rather than as a general statement about the clinical importance of these modalities.

---

# 8.3.10 Relation to the Missing-Radiograph Problem

The original motivation for this experiment was the naturally incomplete availability of radiographs in the COde dataset.

The radiograph-missing scenario achieved:

```text
Macro F1 = 0.6890
Micro F1 = 0.8470
AUROC = 0.9670
```

without retraining the model specifically for the missing-radiograph test condition.

This is important because the classifier was trained using stochastic modality dropout and subsequently evaluated under a deterministic radiograph-missing configuration.

The result demonstrates that the proposed fusion strategy can continue producing predictions when radiographs are unavailable.

At the same time, the decrease in Macro F1 relative to the complete-case condition indicates that robustness is not equivalent to complete invariance. Missing radiographs still reduce classification performance, although the model retains strong overall predictive discrimination.

---

# 8.3.11 Important Experimental Limitation

The missing-modality evaluation was performed on the **complete-case test population** inherited from the Milestone 7 fusion experiment.

Therefore, the test samples all originally contained all three modalities.

The missing modalities in this experiment were simulated by masking the corresponding frozen representations at inference time.

Consequently, this experiment should be described as:

> **controlled missing-modality robustness evaluation**

rather than as a direct evaluation on naturally incomplete test visits.

This distinction is important for the final thesis methodology.

The experiment establishes whether the fusion model can tolerate missing modalities under controlled conditions, while a separate analysis is required to determine how performance behaves on naturally occurring incomplete cases.

---

# 8.3.12 Reproducibility

The complete experiment can be reproduced using:

### Training

```bash
python -m src.missing_modality.train_robust_fusion
```

### Testing

```bash
python -m src.missing_modality.test_robust_fusion
```

Training output:

```text
results/milestone8_missing_modality/04_robust_fusion_training/
```

Test output:

```text
results/milestone8_missing_modality/05_robust_fusion_test/
```

The experiment uses:

```text
Seed = 42
Modality dropout probability = 0.30
```

and keeps the SSL encoders frozen.

---

# 8.3.13 Milestone Status

**Milestone 8.3 — COMPLETE**

Completed components:

* [x] Robust fusion dataset
* [x] Dynamic modality dropout
* [x] Explicit modality-presence mask
* [x] Robust fusion training
* [x] Validation-based checkpoint selection
* [x] Complete-case test evaluation
* [x] Single-modality missingness evaluation
* [x] Multi-modality missingness evaluation
* [x] Reproducible experiment artifacts

Primary result:

> The frozen SSL-based robust fusion classifier maintained strong predictive performance under controlled missing-modality conditions, with relatively small degradation when photographs were unavailable, moderate degradation when radiographs were unavailable, and substantial degradation when clinical text was unavailable.

This provides experimental evidence that modality-dropout-based fusion can improve the operational robustness of multimodal dental diagnosis under incomplete modality availability.


# 08 — Missing Modality Robustness: Standard Fusion Ablation

## Milestone 8.4 — Controlled Ablation Study

**Project:** Self-Supervised Multimodal Representation Learning for Robust Dental Diagnosis with Naturally Missing Radiographs
**Dataset:** COde
**Phase:** Missing-Modality Robustness
**Milestone:** 8.4
**Seed:** 42
**Status:** Completed

---

## 1. Objective

The purpose of Milestone 8.4 is to establish a controlled experimental comparison between:

1. **Standard Fusion**
2. **Robust Fusion**

The objective is to determine whether the robustness observed in Milestone 8.3 is attributable to the explicit robustness mechanisms of the proposed model rather than to differences in architecture, data, optimization, or experimental protocol.

The comparison is therefore designed as a controlled ablation.

The same frozen SSL representations, patient-level population, train/validation/test split, fusion architecture, optimizer, loss function, batch size, learning rate, weight decay, dropout, and model-selection criterion are used for both models.

The intended differences are:

### Standard Fusion

* No modality dropout during training.
* No explicit modality-presence information.
* Trained only with complete multimodal representations.
* Missing modalities are simulated only during testing by zeroing their representations.

### Robust Fusion

* Modality dropout during training.
* Explicit modality-presence mask.
* The model receives information about which modalities are available.
* Missing modalities are simulated during testing using the same controlled scenarios.

This design isolates the contribution of robustness-oriented training and modality-awareness.

---

# 2. Experimental Context

Milestone 8 builds on the frozen multimodal representations extracted during the previous fusion milestones.

The representations are:

| Modality      | Representation Dimension |
| ------------- | -----------------------: |
| Photograph    |                     2048 |
| Radiograph    |                     2048 |
| Clinical Text |                      768 |

The representations are produced by the previously trained self-supervised multimodal encoders and are treated as frozen inputs during the fusion experiments.

The complete-case test population contains:

**633 visits**

This population is identical for Standard Fusion and Robust Fusion.

Therefore, differences between the two methods cannot be attributed to different test populations.

---

# 3. Standard Fusion Ablation

## 3.1 Model Architecture

The Standard Fusion experiment uses the same underlying fusion architecture as RobustFusion.

Each modality is independently projected:

```text
Photograph       2048 → 512
Radiograph       2048 → 512
Clinical Text     768 → 512
```

The three projected representations are concatenated:

```text
512 + 512 + 512 = 1536
```

and passed through:

```text
1536 → 512 → 6
```

with ReLU activation and classifier dropout.

For the Standard Fusion control, all modalities are assumed to be available during training.

No modality-presence information is provided to the classifier.

---

## 3.2 Training Configuration

| Parameter                         | Value               |
| --------------------------------- | ------------------- |
| SSL representations               | Frozen              |
| Training population               | Complete-case       |
| Batch size                        | 64                  |
| Epochs                            | 50                  |
| Learning rate                     | 1e-4                |
| Weight decay                      | 1e-4                |
| Classifier dropout                | 0.3                 |
| Loss                              | BCEWithLogitsLoss   |
| Optimizer                         | AdamW               |
| Random seed                       | 42                  |
| Model selection                   | Validation Macro-F1 |
| Test set used for training        | No                  |
| Test set used for model selection | No                  |

The best Standard Fusion checkpoint was selected using validation Macro-F1.

### Best checkpoint

| Property            |  Value |
| ------------------- | -----: |
| Best epoch          |     46 |
| Validation Macro-F1 | 0.7435 |

The resulting checkpoint was saved under:

```text
results/milestone8_missing_modality/
└── 06_robustness_ablation/
    └── 02_standard_fusion/
        └── best_model.pt
```

---

# 4. Controlled Missing-Modality Evaluation

The same seven evaluation scenarios are applied to both Standard Fusion and Robust Fusion.

The scenarios are generated from the complete-case test population.

For each scenario, the corresponding modality representation is zeroed.

For Robust Fusion, the modality-presence mask is simultaneously updated.

For Standard Fusion, no modality-presence information is provided.

The scenarios are:

| Scenario                   | Image | Radiograph | Text |
| -------------------------- | ----: | ---------: | ---: |
| Complete                   |     ✓ |          ✓ |    ✓ |
| Image missing              |     ✗ |          ✓ |    ✓ |
| Radiograph missing         |     ✓ |          ✗ |    ✓ |
| Text missing               |     ✓ |          ✓ |    ✗ |
| Image + Radiograph missing |     ✗ |          ✗ |    ✓ |
| Image + Text missing       |     ✗ |          ✓ |    ✗ |
| Radiograph + Text missing  |     ✓ |          ✗ |    ✗ |

All scenarios contain exactly:

**633 test samples**

---

# 5. Standard Fusion Results

The complete Standard Fusion evaluation produced the following results.

| Scenario                   | Macro-F1 | Micro-F1 |  AUROC | Accuracy |
| -------------------------- | -------: | -------: | -----: | -------: |
| Complete                   |   0.7691 |   0.8745 | 0.9641 |   0.8073 |
| Image missing              |   0.7729 |   0.8704 | 0.9644 |   0.7946 |
| Radiograph missing         |   0.6850 |   0.8242 | 0.9608 |   0.7330 |
| Text missing               |   0.3635 |   0.5492 | 0.7526 |   0.5039 |
| Image + Radiograph missing |   0.7071 |   0.8368 | 0.9527 |   0.7409 |
| Image + Text missing       |   0.3021 |   0.5344 | 0.6777 |   0.4550 |
| Radiograph + Text missing  |   0.2236 |   0.2171 | 0.7227 |   0.1137 |

---

# 6. Robust Fusion Results

The Robust Fusion model from Milestones 8.3.2–8.3.3 was evaluated using exactly the same test population and missing-modality scenarios.

| Scenario                   | Macro-F1 | Micro-F1 |  AUROC | Accuracy |
| -------------------------- | -------: | -------: | -----: | -------: |
| Complete                   |   0.7756 |   0.8685 | 0.9658 |   0.8025 |
| Image missing              |   0.7853 |   0.8652 | 0.9657 |   0.7978 |
| Radiograph missing         |   0.6890 |   0.8470 | 0.9670 |   0.7741 |
| Text missing               |   0.5251 |   0.7261 | 0.8695 |   0.6477 |
| Image + Radiograph missing |   0.7110 |   0.8458 | 0.9622 |   0.7678 |
| Image + Text missing       |   0.4066 |   0.6326 | 0.8197 |   0.5671 |
| Radiograph + Text missing  |   0.4039 |   0.6918 | 0.8342 |   0.5972 |

---

# 7. Standard vs Robust Fusion

The central comparison of Milestone 8.4 is the difference between Standard Fusion and Robust Fusion under identical controlled missing-modality conditions.

## 7.1 Macro-F1 and AUROC

| Missing-Modality Scenario     | Standard Macro-F1 | Robust Macro-F1 |  Δ Macro-F1 | Standard AUROC | Robust AUROC |     Δ AUROC |
| ----------------------------- | ----------------: | --------------: | ----------: | -------------: | -----------: | ----------: |
| Complete                      |            0.7691 |      **0.7756** | **+0.0065** |         0.9641 |   **0.9658** |     +0.0016 |
| Image missing                 |            0.7729 |      **0.7853** | **+0.0124** |         0.9644 |   **0.9657** |     +0.0013 |
| Radiograph missing            |            0.6850 |      **0.6890** | **+0.0040** |         0.9608 |   **0.9670** | **+0.0062** |
| **Text missing**              |            0.3635 |      **0.5251** | **+0.1617** |         0.7526 |   **0.8695** | **+0.1169** |
| Image + Radiograph missing    |            0.7071 |      **0.7110** |     +0.0039 |         0.9527 |   **0.9622** |     +0.0096 |
| **Image + Text missing**      |            0.3021 |      **0.4066** | **+0.1045** |         0.6777 |   **0.8197** | **+0.1420** |
| **Radiograph + Text missing** |            0.2236 |      **0.4039** | **+0.1803** |         0.7227 |   **0.8342** | **+0.1115** |

---

## 7.2 Micro-F1 and Accuracy

| Missing-Modality Scenario     | Standard Micro-F1 | Robust Micro-F1 |  Δ Micro-F1 | Standard Accuracy | Robust Accuracy |  Δ Accuracy |
| ----------------------------- | ----------------: | --------------: | ----------: | ----------------: | --------------: | ----------: |
| Complete                      |            0.8745 |          0.8685 |     −0.0060 |        **0.8073** |          0.8025 |     −0.0047 |
| Image missing                 |        **0.8704** |          0.8652 |     −0.0051 |            0.7946 |      **0.7978** |     +0.0032 |
| Radiograph missing            |            0.8242 |      **0.8470** | **+0.0229** |            0.7330 |      **0.7741** | **+0.0411** |
| **Text missing**              |            0.5492 |      **0.7261** | **+0.1768** |            0.5039 |      **0.6477** | **+0.1438** |
| Image + Radiograph missing    |            0.8368 |      **0.8458** |     +0.0090 |            0.7409 |      **0.7678** |     +0.0269 |
| **Image + Text missing**      |            0.5344 |      **0.6326** | **+0.0983** |            0.4550 |      **0.5671** | **+0.1122** |
| **Radiograph + Text missing** |            0.2171 |      **0.6918** | **+0.4747** |            0.1137 |      **0.5972** | **+0.4834** |

---

# 8. Main Findings

## 8.1 Complete-case performance

Under complete availability, the two models perform very similarly.

Macro-F1:

```text
Standard Fusion : 0.7691
Robust Fusion   : 0.7756
Improvement     : +0.0065
```

AUROC is also nearly identical:

```text
Standard Fusion : 0.9641
Robust Fusion   : 0.9658
```

This is important because the primary advantage of Robust Fusion is not simply higher complete-case performance.

The models have comparable performance when all modalities are available.

Therefore, the subsequent differences under missing-modality conditions are more directly interpretable as robustness effects.

---

## 8.2 Image missing

Removing the photograph modality has only a small effect on either model.

Robust Fusion achieves:

```text
Macro-F1 = 0.7853
AUROC    = 0.9657
```

compared with:

```text
Macro-F1 = 0.7729
AUROC    = 0.9644
```

for Standard Fusion.

The improvement in Macro-F1 is:

```text
+0.0124
```

This indicates that the model can maintain strong performance when the photograph modality is unavailable.

---

## 8.3 Radiograph missing

When radiographs are removed, Robust Fusion again performs slightly better.

```text
Macro-F1:
Standard = 0.6850
Robust   = 0.6890

AUROC:
Standard = 0.9608
Robust   = 0.9670
```

The AUROC improvement is:

```text
+0.0062
```

The Micro-F1 improvement is larger:

```text
0.8242 → 0.8470
Δ = +0.0229
```

---

## 8.4 Text missing

The most important single-modality robustness result occurs when clinical text is unavailable.

Standard Fusion experiences a substantial degradation:

```text
Macro-F1 = 0.3635
Micro-F1 = 0.5492
AUROC    = 0.7526
Accuracy = 0.5039
```

Robust Fusion performs considerably better:

```text
Macro-F1 = 0.5251
Micro-F1 = 0.7261
AUROC    = 0.8695
Accuracy = 0.6477
```

The improvements are:

```text
Macro-F1 : +0.1617
Micro-F1 : +0.1768
AUROC    : +0.1169
Accuracy : +0.1438
```

This is strong evidence that robustness-oriented training substantially improves the model's ability to operate without the clinical text modality.

---

# 9. Multiple Missing Modalities

The strongest evidence for Robust Fusion appears when two modalities are simultaneously unavailable.

## 9.1 Image + Text Missing

Standard Fusion:

```text
Macro-F1 = 0.3021
Micro-F1 = 0.5344
AUROC    = 0.6777
Accuracy = 0.4550
```

Robust Fusion:

```text
Macro-F1 = 0.4066
Micro-F1 = 0.6326
AUROC    = 0.8197
Accuracy = 0.5671
```

Improvements:

```text
Macro-F1 : +0.1045
Micro-F1 : +0.0983
AUROC    : +0.1420
Accuracy : +0.1122
```

---

## 9.2 Radiograph + Text Missing

This is the most severe controlled missing-modality scenario because only the photograph modality remains available.

Standard Fusion degrades dramatically:

```text
Macro-F1 = 0.2236
Micro-F1 = 0.2171
AUROC    = 0.7227
Accuracy = 0.1137
```

Robust Fusion maintains substantially stronger performance:

```text
Macro-F1 = 0.4039
Micro-F1 = 0.6918
AUROC    = 0.8342
Accuracy = 0.5972
```

The differences are:

```text
Macro-F1 : +0.1803
Micro-F1 : +0.4747
AUROC    : +0.1115
Accuracy : +0.4834
```

The Micro-F1 and Accuracy improvements are particularly large.

This result suggests that the robustness mechanisms are especially valuable when the model is forced to operate with only a single remaining modality.

---

# 10. Overall Interpretation

The ablation study supports the hypothesis that explicit robustness mechanisms improve multimodal fusion under missing-modality conditions.

The most important observation is that the benefit of Robust Fusion is not primarily visible in the complete-case setting.

Instead, the performance gap becomes increasingly pronounced as informative modalities are removed.

In particular:

1. **Complete-case performance remains comparable.**
2. **Image missing causes only a small difference.**
3. **Radiograph missing produces a moderate robustness benefit.**
4. **Text missing produces a substantial improvement.**
5. **Two-modality missing scenarios produce the largest improvements.**

The strongest result is observed when both radiographs and clinical text are missing:

```text
Standard Macro-F1 = 0.2236
Robust Macro-F1   = 0.4039
```

and:

```text
Standard Micro-F1 = 0.2171
Robust Micro-F1   = 0.6918
```

Thus, Robust Fusion demonstrates substantially better degradation resistance under severe modality loss.

---

# 11. Experimental Validity

The following controls were maintained:

* Same SSL representation root.
* Same frozen representations.
* Same complete-case test population.
* Same 633 test samples.
* Same train/validation/test split.
* Same fusion dimensions.
* Same hidden dimension.
* Same optimizer.
* Same learning rate.
* Same weight decay.
* Same loss function.
* Same batch size.
* Same classifier dropout.
* Same random seed.
* Same validation Macro-F1 model-selection criterion.
* Test set excluded from training.
* Test set excluded from model selection.
* Same seven missing-modality scenarios.

Therefore, the comparison is a controlled ablation rather than a comparison between independently optimized systems.

---

# 12. Reproducibility

## Standard Fusion Training

```bash
python -m src.missing_modality.train_standard_fusion_ablation
```

Output:

```text
results/milestone8_missing_modality/
└── 06_robustness_ablation/
    └── 02_standard_fusion/
        ├── best_model.pt
        ├── history.json
        └── config.json
```

## Standard Fusion Test

```bash
python -m src.missing_modality.test_standard_fusion_ablation
```

Output:

```text
results/milestone8_missing_modality/
└── 06_robustness_ablation/
    └── 03_standard_fusion_test/
        ├── test_results.csv
        ├── test_results.json
        └── config.json
```

## Robust Fusion Test

```bash
python -m src.missing_modality.test_robust_fusion
```

Output:

```text
results/milestone8_missing_modality/
└── 05_robust_fusion_test/
    ├── test_results.csv
    ├── test_results.json
    └── config.json
```

---

# 13. Final Milestone 8.4 Conclusion

Milestone 8.4 established the required controlled ablation baseline for Robust Fusion.

The Standard Fusion model achieves competitive performance when all modalities are available, but its performance degrades substantially under missing-modality conditions, particularly when clinical text is unavailable or when multiple modalities are simultaneously missing.

Robust Fusion consistently provides better Macro-F1 and AUROC across all controlled missing-modality scenarios and shows particularly large gains under severe modality loss.

The results therefore support the central robustness claim of the proposed fusion strategy:

> Robust multimodal fusion is substantially more resilient to missing modalities than standard fusion when evaluated under controlled modality-loss conditions.

The strongest evidence is obtained in the two-modality missing scenario:

```text
Radiograph + Text missing

Macro-F1:
Standard Fusion = 0.2236
Robust Fusion   = 0.4039

Micro-F1:
Standard Fusion = 0.2171
Robust Fusion   = 0.6918
```

This milestone therefore provides the controlled experimental evidence required to distinguish **standard multimodal fusion** from the proposed **robust missing-modality-aware fusion strategy**.
